---
title: Creates Target Variable
---

inputs: 
- data_processed/county_pop_historical.xlsx     # column data with historical population density by county
                                                # created by population_expansion.ipynb
                                                   

outputs:  
- data_processed/target/population_dfs.h5     # dictionary with Change in pop density by County and decade
- data_processed/target/population_dfs.h5     # dictionary with Pop density by County and decade

In [1]:
import pandas as pd

In [2]:
#population = pd.read_excel('data/county2010_hist_pops.xlsx')
population = pd.read_excel('data_processed/county_pop_historical.xlsx')


In [3]:
population.head(2)

,GEO_ID,STATE,COUNTY,NAME,LSAD,CENSUSAREA,geometry,GEOID10,epop1790,epop1800,...,edens1910,edens1920,edens1930,edens1940,edens1950,edens1960,edens1970,edens1980,edens1990,edens2000
0,0500000US01029,1,29,Cleburne,County,560.100,"POLYGON ((-85.3887171312565 33.9130442466707, ...",1029,0.0,0.0,...,23.884703,23.852883,22.990537,24.333155,21.253348,19.480450,19.632209,22.487056,22.728084,25.215140
1,0500000US01031,1,31,Coffee,County,678.972,"POLYGON ((-86.030441 31.618938999999997, -86.0...",1031,0.0,0.0,...,38.468449,44.287541,47.948958,47.110927,45.244870,45.043095,51.359997,56.751972,59.266067,64.244181


In [4]:
# Calculate change in density columns (DdensYYYY)
year_columns = [col for col in population.columns if col.startswith('edens')]
for i in range(1, len(year_columns)):
    curr_col = year_columns[i]
    prev_col = year_columns[i-1]
    if curr_col.startswith('edens') and prev_col.startswith('edens'):
        year = ''.join(filter(str.isdigit, curr_col))
        new_col = f'Ddens{year}'
        population[new_col] = population[curr_col] - population[prev_col]

In [5]:
# quick visual check to make sure correctly calculated
population['Ddens1790']=0
population[['GEO_ID', 'NAME', 'edens1890', 'edens1900', 'Ddens1900']].head()

,GEO_ID,NAME,edens1890,edens1900,Ddens1900
0,0500000US01029,Cleburne,23.519217,23.453914,-0.065303
1,0500000US01031,Coffee,17.924156,30.887872,12.963716
2,0500000US01037,Coosa,24.437344,24.803207,0.365863
3,0500000US01039,Covington,7.333052,14.892436,7.559384
4,0500000US01041,Crenshaw,25.301577,32.284900,6.983323


In [6]:
# insert the Ddens1790 column in the right place
population = population[['GEO_ID', 'STATE', 'COUNTY', 'NAME', 'CENSUSAREA', 'geometry',
       'GEOID10', 'epop1790', 'epop1800', 'epop1810', 'epop1820', 'epop1830',
       'epop1840', 'epop1850', 'epop1860', 'epop1870', 'epop1880', 'epop1890',
       'epop1900', 'epop1910', 'epop1920', 'epop1930', 'epop1940', 'epop1950',
       'epop1960', 'epop1970', 'epop1980', 'epop1990', 'epop2000', 'pop2010',
       'edens1790', 'edens1800', 'edens1810', 'edens1820', 'edens1830',
       'edens1840', 'edens1850', 'edens1860', 'edens1870', 'edens1880',
       'edens1890', 'edens1900', 'edens1910', 'edens1920', 'edens1930',
       'edens1940', 'edens1950', 'edens1960', 'edens1970', 'edens1980',
       'edens1990', 'edens2000', 'Ddens1790', 'Ddens1800', 'Ddens1810', 'Ddens1820',
       'Ddens1830', 'Ddens1840', 'Ddens1850', 'Ddens1860', 'Ddens1870',
       'Ddens1880', 'Ddens1890', 'Ddens1900', 'Ddens1910', 'Ddens1920',
       'Ddens1930', 'Ddens1940', 'Ddens1950', 'Ddens1960', 'Ddens1970',
       'Ddens1980', 'Ddens1990', 'Ddens2000']]

## Create Target DataFrame/Dictionary

In [7]:
# Create a dictionary of dataframes for each year
population_dfs = {}
year_columns = [col for col in population.columns if col.startswith('Ddens') or col.startswith('edens')]
for year_col in year_columns:
    year = int(''.join(filter(str.isdigit, year_col)))
    population_dfs[year] = population[['GEOID10', 'STATE', 'COUNTY', year_col]].rename(columns={year_col: 'Change_density'})

In [8]:
population_dfs[1830].head(2)

,GEOID10,STATE,COUNTY,Change_density
0,1029,1,29,0.000000
1,1031,1,31,0.903562


In [9]:
def save_dfs_h5(population_dfs, filename='data_processed/target/population_dfs.h5'):
    """
    Save the population_dfs dictionary of DataFrames to an HDF5 file.

    Parameters:
        population_dfs (dict): Dictionary of year -> pandas DataFrame.
        filename (str): Output HDF5 filename.
    """
    with pd.HDFStore(filename, mode='w') as store:
        for year, df in population_dfs.items():
            store.put(str(year), df)
    print(f"Saved population_dfs to {filename}")

# Example usage:
# save_population_dfs_h5(population_dfs)

In [10]:
population_dfs.keys()

dict_keys([1790, 1800, 1810, 1820, 1830, 1840, 1850, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000])

In [11]:
save_dfs_h5(population_dfs)

Saved population_dfs to data_processed/target/population_dfs.h5


c:\Users\jonat\miniforge3\envs\gnn_env\Lib\site-packages\tables\path.py:146: NaturalNameWarning: object name is not a valid Python identifier: '1790'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
c:\Users\jonat\miniforge3\envs\gnn_env\Lib\site-packages\tables\path.py:146: NaturalNameWarning: object name is not a valid Python identifier: '1800'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
c:\Users\jonat\miniforge3\envs\gnn_env\Lib\site-packages\tables\path.py:146: NaturalNameWarning: object name is not a valid Python identifier: '1810'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` wil